In [ ]:
!pip install -q groq sentence-transformers faiss-cpu pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 71.5 MB/s eta 0:00:00


In [ ]:
import json
import time
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from groq import Groq
from google.colab import userdata

In [ ]:
groq_api_key = userdata.get("GROQ_API_KEY")
if groq_api_key:
    print("Groq API key loaded successfully.")
else:
    print("Groq API key not found.")

Groq API key loaded successfully.


In [ ]:
client = Groq(
    api_key=groq_api_key
)

In [ ]:
LARGE_MODEL = "openai/gpt-oss-120b"
SMALL_MODEL = "openai/gpt-oss-20b"
print("Large model:", LARGE_MODEL)
print("Small model:", SMALL_MODEL)

Large model: openai/gpt-oss-120b
Small model: openai/gpt-oss-20b


In [ ]:
!wget -q https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip
!unzip -qo scifact.zip

In [ ]:
!find scifact -maxdepth 2 -type f

scifact/qrels/test.tsv
scifact/qrels/train.tsv
scifact/corpus.jsonl
scifact/queries.jsonl


In [ ]:
corpus = {}
with open("scifact/corpus.jsonl", "r") as f:
    for line in f:
        doc = json.loads(line)
        corpus[str(doc["_id"])] = {
            "title": doc["title"],
            "text": doc["text"]
        }
print("Corpus documents:", len(corpus))

Corpus documents: 5183


In [ ]:
documents_df = pd.DataFrame([
    {
        "doc_id": doc_id,
        "title": doc["title"],
        "text": doc["text"]
    }
    for doc_id, doc in corpus.items()
])
documents_df["content"] = (
    documents_df["title"]
    + "\n\n"
    + documents_df["text"]
)
print(
    "Documents:",
    len(documents_df)
)

Documents: 5183


In [ ]:
queries = {}
with open("scifact/queries.jsonl", "r") as f:
    for line in f:
        query = json.loads(line)
        queries[str(query["_id"])] = query["text"]
print("Queries:", len(queries))

Queries: 1109


In [ ]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)
print(
    "Embedding dimension:",
    embedding_model.get_sentence_embedding_dimension()
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimension: 384


/tmp/ipykernel_4226/990150634.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_model.get_sentence_embedding_dimension()


In [ ]:
document_embeddings = embedding_model.encode(
    documents_df["content"].tolist(),batch_size=32,
    show_progress_bar=True,normalize_embeddings=True
).astype("float32")
index = faiss.IndexFlatIP(document_embeddings.shape[1])
index.add(document_embeddings)
doc_id_mapping = (documents_df["doc_id"].tolist())
print("FAISS documents:",index.ntotal)

Batches:   0%|          | 0/162 [00:00<?, ?it/s]

FAISS documents: 5183


In [ ]:
def search_documents(query,top_k=5):
    query_embedding = embedding_model.encode(
        [query],normalize_embeddings=True
    ).astype("float32")
    scores, indices = index.search(
        query_embedding,top_k
    )
    results = []
    for score, idx in zip(
        scores[0],indices[0]
    ):
        doc_id = doc_id_mapping[idx]
        document = documents_df[
            documents_df["doc_id"] == doc_id
        ].iloc[0]
        results.append({
            "doc_id": doc_id,
            "score": float(score),
            "title": document["title"],
            "text": document["text"]
        })
    return results

In [ ]:
def build_context(results):
    context_parts = []
    for i, result in enumerate(
        results,
        start=1
    ):
        context_parts.append(
            f"[Document {i}]\n"
            f"Title: {result['title']}\n"
            f"Content: {result['text']}"
        )
    return "\n\n".join(
        context_parts
    )

In [ ]:
def generate_rag_answer(
    query,model,top_k=5
):
    start_time = time.perf_counter()
    retrieved_docs = search_documents(
        query,top_k=top_k
    )
    retrieval_end = time.perf_counter()
    context = build_context(
        retrieved_docs
    )
    system_prompt = """
You are a scientific question-answering assistant.
Answer the user's question using only the provided context.
Rules:
- Do not use outside knowledge.
- Do not invent facts.
- If the context does not contain enough information, say so.
- Give a concise and scientifically grounded answer.
"""
    user_prompt = f"""
Context:
{context}
Question:
{query}
Answer:
"""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],temperature=0
    )
    end_time = time.perf_counter()
    usage = response.usage
    input_tokens = getattr(
        usage,"prompt_tokens",0
    )
    output_tokens = getattr(
        usage,"completion_tokens",0
    )
    return {
        "query": query,"model": model,
        "answer": response.choices[0].message.content,
        "retrieved_documents": retrieved_docs,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": (
            input_tokens + output_tokens
        ),
        "retrieval_latency_sec":retrieval_end - start_time,
        "generation_latency_sec":end_time - retrieval_end,
        "total_latency_sec":end_time - start_time
    }

In [ ]:
queries_df = pd.DataFrame([
    {
        "query_id": query_id,"query": query_text
    }
    for query_id, query_text in queries.items()
])
print("Queries DataFrame created:", queries_df.shape)
queries_df.head()

Queries DataFrame created: (1109, 2)


,query_id,query
0,0,0-dimensional biomaterials lack inductive prop...
1,2,1 in 5 million in UK have abnormal PrP positiv...
2,4,1-1% of colorectal cancer patients are diagnos...
3,6,10% of sudden infant death syndrome (SIDS) dea...
4,9,32% of liver transplantation programs required...


In [ ]:
test_query = queries_df["query"].iloc[0]
large_result = generate_rag_answer(
    query=test_query,
    model=LARGE_MODEL,
    top_k=5
)
print("QUERY:")
print(test_query)
print("\nANSWER:")
print(large_result["answer"])
print("\nMODEL:")
print(large_result["model"])
print("\nINPUT TOKENS:")
print(large_result["input_tokens"])
print("\nOUTPUT TOKENS:")
print(large_result["output_tokens"])
print("\nTOTAL TOKENS:")
print(large_result["total_tokens"])
print("\nLATENCY:")
print(round(large_result["total_latency_sec"],3),"seconds")

QUERY:
0-dimensional biomaterials lack inductive properties.

ANSWER:
The provided documents do not contain information about 0‑dimensional biomaterials or their inductive properties, so I cannot answer the statement from the given context.

MODEL:
openai/gpt-oss-120b

INPUT TOKENS:
947

OUTPUT TOKENS:
142

TOTAL TOKENS:
1089

LATENCY:
0.954 seconds


In [ ]:
small_result = generate_rag_answer(
    query=test_query,model=SMALL_MODEL,top_k=5
)
print("QUERY:")
print(test_query)
print("\nANSWER:")
print(small_result["answer"])
print("\nMODEL:")
print(small_result["model"])
print("\nINPUT TOKENS:")
print(small_result["input_tokens"])
print("\nOUTPUT TOKENS:")
print(small_result["output_tokens"])
print("\nTOTAL TOKENS:")
print(small_result["total_tokens"])
print("\nLATENCY:")
print(round(small_result["total_latency_sec"],3),"seconds")

QUERY:
0-dimensional biomaterials lack inductive properties.

ANSWER:
The provided context does not contain information about 0‑dimensional biomaterials or their inductive properties.

MODEL:
openai/gpt-oss-20b

INPUT TOKENS:
947

OUTPUT TOKENS:
157

TOTAL TOKENS:
1104

LATENCY:
0.702 seconds


In [ ]:
print("QUERY:")
print(test_query)

print("\n" + "-" * 80)
print("GPT-OSS-120B")
print("-" * 80)
print(large_result["answer"])

print("\n" + "-" * 80)
print("GPT-OSS-20B")
print("-" * 80)
print(small_result["answer"])

QUERY:
0-dimensional biomaterials lack inductive properties.

--------------------------------------------------------------------------------
GPT-OSS-120B
--------------------------------------------------------------------------------
The provided documents do not contain information about 0‑dimensional biomaterials or their inductive properties, so I cannot answer the statement from the given context.

--------------------------------------------------------------------------------
GPT-OSS-20B
--------------------------------------------------------------------------------
The provided context does not contain information about 0‑dimensional biomaterials or their inductive properties.


In [ ]:
MODEL_PRICING = {
    "openai/gpt-oss-20b": {
        "input_per_million": 0.075,"output_per_million": 0.30
    },
    "openai/gpt-oss-120b": {
        "input_per_million": 0.15,"output_per_million": 0.60
    }
}

In [ ]:
def calculate_model_cost(model,input_tokens,output_tokens):
    pricing = MODEL_PRICING[model]
    input_cost = (input_tokens / 1_000_000
    ) * pricing["input_per_million"]
    output_cost = (output_tokens / 1_000_000
    ) * pricing["output_per_million"]
    return input_cost + output_cost

In [ ]:
large_cost = calculate_model_cost(
    LARGE_MODEL,large_result["input_tokens"],
    large_result["output_tokens"]
)
small_cost = calculate_model_cost(
    SMALL_MODEL,small_result["input_tokens"],
    small_result["output_tokens"]
)
print("GPT-OSS-120B cost:",f"${large_cost:.6f}")
print("GPT-OSS-20B cost:",f"${small_cost:.6f}")

GPT-OSS-120B cost: $0.000227
GPT-OSS-20B cost: $0.000118


In [ ]:
benchmark_queries = (queries_df["query"].iloc[:20].tolist())
print("Benchmark queries:",len(benchmark_queries))

Benchmark queries: 20


In [ ]:
benchmark_results = []
for i, query in enumerate(
    benchmark_queries,start=1
):
    print(f"Processing query {i}/20...")
    large = generate_rag_answer(
        query=query,model=LARGE_MODEL,top_k=5
    )
    small = generate_rag_answer(
        query=query,model=SMALL_MODEL,top_k=5
    )
    large_cost = calculate_model_cost(
        LARGE_MODEL,large["input_tokens"],large["output_tokens"]
    )
    small_cost = calculate_model_cost(
        SMALL_MODEL,small["input_tokens"],small["output_tokens"]
    )
    benchmark_results.append({
        "query": query,
        "large_input_tokens":large["input_tokens"],
        "large_output_tokens":large["output_tokens"],
        "large_total_tokens":large["total_tokens"],
        "large_latency_sec":large["total_latency_sec"],
        "large_cost":large_cost,
        "large_answer":large["answer"],
        "small_input_tokens":small["input_tokens"],
        "small_output_tokens":small["output_tokens"],
        "small_total_tokens":small["total_tokens"],
        "small_latency_sec":small["total_latency_sec"],
        "small_cost":small_cost,
        "small_answer":small["answer"]
    })

Processing query 1/20...
Processing query 2/20...
Processing query 3/20...
Processing query 4/20...
Processing query 5/20...
Processing query 6/20...
Processing query 7/20...
Processing query 8/20...
Processing query 9/20...
Processing query 10/20...
Processing query 11/20...
Processing query 12/20...
Processing query 13/20...
Processing query 14/20...
Processing query 15/20...
Processing query 16/20...
Processing query 17/20...
Processing query 18/20...
Processing query 19/20...
Processing query 20/20...


In [ ]:
benchmark_df = pd.DataFrame(benchmark_results)
print("Benchmark completed.")
print("Rows:",len(benchmark_df))

Benchmark completed.
Rows: 20


In [ ]:
benchmark_summary = pd.DataFrame([
    {
        "Metric": "Average Latency (sec)",
        "GPT-OSS-20B":
            benchmark_df[
                "small_latency_sec"
            ].mean(),
        "GPT-OSS-120B":
            benchmark_df[
                "large_latency_sec"
            ].mean()
    },
    {
        "Metric": "Average Input Tokens",
        "GPT-OSS-20B":
            benchmark_df[
                "small_input_tokens"
            ].mean(),
        "GPT-OSS-120B":
            benchmark_df[
                "large_input_tokens"
            ].mean()
    },
    {
        "Metric": "Average Output Tokens",
        "GPT-OSS-20B":
            benchmark_df[
                "small_output_tokens"
            ].mean(),
        "GPT-OSS-120B":
            benchmark_df[
                "large_output_tokens"
            ].mean()
    },
    {
        "Metric": "Average Total Tokens",
        "GPT-OSS-20B":
            benchmark_df[
                "small_total_tokens"
            ].mean(),
        "GPT-OSS-120B":
            benchmark_df[
                "large_total_tokens"
            ].mean()
    },
    {
        "Metric": "Average Cost ($)",
        "GPT-OSS-20B":
            benchmark_df[
                "small_cost"
            ].mean(),
        "GPT-OSS-120B":
            benchmark_df[
                "large_cost"
            ].mean()
    }
])
benchmark_summary

,Metric,GPT-OSS-20B,GPT-OSS-120B
0,Average Latency (sec),0.848552,12.358472
1,Average Input Tokens,1911.850000,1911.850000
2,Average Output Tokens,232.900000,234.050000
3,Average Total Tokens,2144.750000,2145.900000
4,Average Cost ($),0.000213,0.000427


In [ ]:
average_small_cost = (benchmark_df["small_cost"].mean())
average_large_cost = (benchmark_df["large_cost"].mean())
cost_difference_pct = (
    1 - average_small_cost /average_large_cost) * 100
print("Average cost reduction using 20B:",round(
        cost_difference_pct,2),"%")

Average cost reduction using 20B: 50.08 %


In [ ]:
benchmark_df.to_csv(
    "model_routing_benchmark.csv",index=False
)
benchmark_summary.to_csv(
    "model_routing_benchmark_summary.csv",index=False
)
print("Model benchmark saved.")

Model benchmark saved.


In [ ]:
def extract_query_features(query, top_k=5):
    """
    Extract inexpensive features used for query difficulty detection.
    """
    query_embedding = embedding_model.encode(
        [query],normalize_embeddings=True
    ).astype("float32")
    scores, indices = index.search(
        query_embedding,top_k
    )
    scores = scores[0]
    query_words = query.split()
    features = {
        "query_word_count": len(query_words),
        "query_char_count": len(query),
        "has_question_mark": int("?" in query),
        "top1_score": float(scores[0]),
        "top2_score": float(scores[1])
            if len(scores) > 1 else 0.0,
        "top1_top2_margin": float(scores[0] - scores[1])
            if len(scores) > 1 else 0.0,
        "mean_top5_score": float(scores.mean()),
        "std_top5_score": float(scores.std())
    }
    return features

In [ ]:
routing_feature_records = []
for query in benchmark_df["query"]:
    features = extract_query_features(
        query,top_k=5
    )
    features["query"] = query
    routing_feature_records.append(features)
routing_features_df = pd.DataFrame(
    routing_feature_records
)
print(
    "Feature dataset shape:",routing_features_df.shape
)
routing_features_df.head()

Feature dataset shape: (20, 9)


,query_word_count,query_char_count,has_question_mark,top1_score,top2_score,top1_top2_margin,mean_top5_score,std_top5_score,query
0,5,53,0,0.345350,0.333349,0.012000,0.313320,0.023501,0-dimensional biomaterials lack inductive prop...
1,10,50,0,0.436206,0.416901,0.019305,0.387583,0.032640,1 in 5 million in UK have abnormal PrP positiv...
2,12,85,0,0.649795,0.602907,0.046887,0.555948,0.061416,1-1% of colorectal cancer patients are diagnos...
3,16,93,0,0.733943,0.601561,0.132381,0.597994,0.072972,10% of sudden infant death syndrome (SIDS) dea...
4,13,99,0,0.767422,0.591280,0.176142,0.512454,0.147609,32% of liver transplantation programs required...


In [ ]:
routing_features_df[
    [
        "query","query_word_count","top1_score",
        "top2_score","top1_top2_margin",
        "mean_top5_score","std_top5_score"
    ]].head(10)

,query,query_word_count,top1_score,top2_score,top1_top2_margin,mean_top5_score,std_top5_score
0,0-dimensional biomaterials lack inductive prop...,5,0.345350,0.333349,0.012000,0.313320,0.023501
1,1 in 5 million in UK have abnormal PrP positiv...,10,0.436206,0.416901,0.019305,0.387583,0.032640
2,1-1% of colorectal cancer patients are diagnos...,12,0.649795,0.602907,0.046887,0.555948,0.061416
3,10% of sudden infant death syndrome (SIDS) dea...,16,0.733943,0.601561,0.132381,0.597994,0.072972
4,32% of liver transplantation programs required...,13,0.767422,0.591280,0.176142,0.512454,0.147609
5,4-PBA treatment decreases endoplasmic reticulu...,14,0.564231,0.480685,0.083546,0.463760,0.055120
6,4-PBA treatment raises endoplasmic reticulum s...,14,0.567920,0.469278,0.098642,0.459759,0.057463
7,40mg/day dosage of folic acid and 2mg/day dosa...,19,0.740401,0.619805,0.120596,0.608169,0.072252
8,5'-nucleotidase metabolizes 6MP.,3,0.556454,0.408765,0.147689,0.414871,0.072442
9,50% of patients exposed to radiation have acti...,13,0.534414,0.499651,0.034763,0.500144,0.020040


In [ ]:
routing_df = routing_features_df.merge(
    benchmark_df[
        [
            "query","small_cost","large_cost",
            "small_latency_sec","large_latency_sec",
            "small_total_tokens","large_total_tokens"
        ]
    ],
    on="query",how="left"
)
print("Routing dataset shape:", routing_df.shape)
routing_df.head()

Routing dataset shape: (20, 15)


,query_word_count,query_char_count,has_question_mark,top1_score,top2_score,top1_top2_margin,mean_top5_score,std_top5_score,query,small_cost,large_cost,small_latency_sec,large_latency_sec,small_total_tokens,large_total_tokens
0,5,53,0,0.345350,0.333349,0.012000,0.313320,0.023501,0-dimensional biomaterials lack inductive prop...,0.000118,0.000227,0.641116,0.831071,1104,1089
1,10,50,0,0.436206,0.416901,0.019305,0.387583,0.032640,1 in 5 million in UK have abnormal PrP positiv...,0.000252,0.000528,0.847703,1.388979,2571,2611
2,12,85,0,0.649795,0.602907,0.046887,0.555948,0.061416,1-1% of colorectal cancer patients are diagnos...,0.000224,0.000478,0.806679,1.200968,2204,2255
3,16,93,0,0.733943,0.601561,0.132381,0.597994,0.072972,10% of sudden infant death syndrome (SIDS) dea...,0.000205,0.000450,0.750483,1.144945,2129,2196
4,13,99,0,0.767422,0.591280,0.176142,0.512454,0.147609,32% of liver transplantation programs required...,0.000205,0.000455,0.896797,10.329811,1923,1997


In [ ]:
routing_df = routing_features_df.merge(
    benchmark_df[
        [
            "query","small_answer","large_answer",
            "small_cost","large_cost","small_latency_sec",
            "large_latency_sec","small_total_tokens",
            "large_total_tokens"
        ]
    ],
    on="query",how="left"
)
print("Routing dataset shape:", routing_df.shape)
routing_df.head()

Routing dataset shape: (20, 17)


,query_word_count,query_char_count,has_question_mark,top1_score,top2_score,top1_top2_margin,mean_top5_score,std_top5_score,query,small_answer,large_answer,small_cost,large_cost,small_latency_sec,large_latency_sec,small_total_tokens,large_total_tokens
0,5,53,0,0.345350,0.333349,0.012000,0.313320,0.023501,0-dimensional biomaterials lack inductive prop...,The provided context does not contain informat...,The provided documents do not contain informat...,0.000118,0.000227,0.641116,0.831071,1104,1089
1,10,50,0,0.436206,0.416901,0.019305,0.387583,0.032640,1 in 5 million in UK have abnormal PrP positiv...,The data from the large‑scale appendix survey ...,The statement is not supported by the data. In...,0.000252,0.000528,0.847703,1.388979,2571,2611
2,12,85,0,0.649795,0.602907,0.046887,0.555948,0.061416,1-1% of colorectal cancer patients are diagnos...,No. In the Dutch nationwide study (1996‑2011) ...,The documents do not support the claim that on...,0.000224,0.000478,0.806679,1.200968,2204,2255
3,16,93,0,0.733943,0.601561,0.132381,0.597994,0.072972,10% of sudden infant death syndrome (SIDS) dea...,"I’m sorry, but the provided documents do not c...",The provided documents do not contain any data...,0.000205,0.000450,0.750483,1.144945,2129,2196
4,13,99,0,0.767422,0.591280,0.176142,0.512454,0.147609,32% of liver transplantation programs required...,"Yes. According to the study, 32 % of liver‑tra...",The provided documents state that 32 percent o...,0.000205,0.000455,0.896797,10.329811,1923,1997


In [ ]:
print(routing_df.columns.tolist())

['query_word_count', 'query_char_count', 'has_question_mark', 'top1_score', 'top2_score', 'top1_top2_margin', 'mean_top5_score', 'std_top5_score', 'query', 'small_answer', 'large_answer', 'small_cost', 'large_cost', 'small_latency_sec', 'large_latency_sec', 'small_total_tokens', 'large_total_tokens']


In [ ]:
from sentence_transformers import util
small_answer_embeddings = embedding_model.encode(
    benchmark_df["small_answer"].tolist(),batch_size=16,
    normalize_embeddings=True,show_progress_bar=True
)
large_answer_embeddings = embedding_model.encode(
    benchmark_df["large_answer"].tolist(),
    batch_size=16,normalize_embeddings=True,
    show_progress_bar=True
)
small_answer_embeddings = np.asarray(
    small_answer_embeddings,dtype="float32"
)
large_answer_embeddings = np.asarray(
    large_answer_embeddings,dtype="float32"
)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
answer_similarity_scores = np.sum(
    small_answer_embeddings * large_answer_embeddings,axis=1
)
routing_df["answer_similarity"] = answer_similarity_scores
routing_df[
    [
        "query","answer_similarity"
    ]].head(10)

,query,answer_similarity
0,0-dimensional biomaterials lack inductive prop...,0.878049
1,1 in 5 million in UK have abnormal PrP positiv...,0.635374
2,1-1% of colorectal cancer patients are diagnos...,0.887530
3,10% of sudden infant death syndrome (SIDS) dea...,0.899179
4,32% of liver transplantation programs required...,0.822267
5,4-PBA treatment decreases endoplasmic reticulu...,0.727522
6,4-PBA treatment raises endoplasmic reticulum s...,0.654069
7,40mg/day dosage of folic acid and 2mg/day dosa...,0.975927
8,5'-nucleotidase metabolizes 6MP.,0.873174
9,50% of patients exposed to radiation have acti...,0.919728


In [ ]:
print(routing_df["answer_similarity"].describe())

count    20.000000
mean      0.867138
std       0.097629
min       0.635374
25%       0.860447
50%       0.896867
75%       0.921866
max       0.975927
Name: answer_similarity, dtype: float64


In [ ]:
routing_df[
    [
        "query","answer_similarity",
        "small_answer","large_answer"
    ]
].sort_values("answer_similarity").head(5)

,query,answer_similarity,small_answer,large_answer
1,1 in 5 million in UK have abnormal PrP positiv...,0.635374,The data from the large‑scale appendix survey ...,The statement is not supported by the data. In...
6,4-PBA treatment raises endoplasmic reticulum s...,0.654069,The provided documents do not mention 4‑phenyl...,The provided documents do not contain any info...
5,4-PBA treatment decreases endoplasmic reticulu...,0.727522,The provided documents do not contain any info...,The provided documents do not contain any info...
11,61% of colorectal cancer patients are diagnose...,0.760664,The provided documents do not contain any info...,The provided documents do not contain informat...
4,32% of liver transplantation programs required...,0.822267,"Yes. According to the study, 32 % of liver‑tra...",The provided documents state that 32 percent o...


In [ ]:
def calculate_grounding_score(
    answer,query,top_k=5
):
    """
    Measures semantic similarity between the answer
    and the retrieved evidence.
    This is a proxy for grounding, not a factuality guarantee.
    """
    retrieved_docs = search_documents(
        query,top_k=top_k
    )
    answer_embedding = embedding_model.encode(
        [answer],normalize_embeddings=True
    ).astype("float32")
    document_texts = [
        doc["title"] + "\n" + doc["text"]
        for doc in retrieved_docs
    ]
    document_embeddings = embedding_model.encode(
        document_texts,normalize_embeddings=True
    ).astype("float32")
    similarities = np.dot(
        document_embeddings,answer_embedding[0]
    )
    return {
        "max_grounding_similarity": float(
            similarities.max()
        ),
        "mean_grounding_similarity": float(
            similarities.mean()
        )
    }

In [ ]:
small_grounding_scores = []
large_grounding_scores = []
for _, row in benchmark_df.iterrows():
    small_score = calculate_grounding_score(
        answer=row["small_answer"],query=row["query"],top_k=5
    )
    large_score = calculate_grounding_score(
        answer=row["large_answer"],query=row["query"],top_k=5
    )
    small_grounding_scores.append(small_score)
    large_grounding_scores.append(large_score)

In [ ]:
routing_df["small_max_grounding"] = [
    x["max_grounding_similarity"]
    for x in small_grounding_scores
]
routing_df["small_mean_grounding"] = [
    x["mean_grounding_similarity"]
    for x in small_grounding_scores
]
routing_df["large_max_grounding"] = [
    x["max_grounding_similarity"]
    for x in large_grounding_scores
]
routing_df["large_mean_grounding"] = [
    x["mean_grounding_similarity"]
    for x in large_grounding_scores
]

In [ ]:
quality_summary = pd.DataFrame({
    "Metric": [
        "Answer similarity",
        "20B max grounding",
        "120B max grounding",
        "20B mean grounding",
        "120B mean grounding"
    ],
    "Average": [
        routing_df["answer_similarity"].mean(),
        routing_df["small_max_grounding"].mean(),
        routing_df["large_max_grounding"].mean(),
        routing_df["small_mean_grounding"].mean(),
        routing_df["large_mean_grounding"].mean()
    ]
})
quality_summary

,Metric,Average
0,Answer similarity,0.867138
1,20B max grounding,0.542507
2,120B max grounding,0.569302
3,20B mean grounding,0.448845
4,120B mean grounding,0.452890


In [ ]:
routing_df[
    [
        "query","answer_similarity",
        "small_max_grounding","large_max_grounding",
        "top1_score","top1_top2_margin"
    ]
].sort_values("answer_similarity").head(10)

,query,answer_similarity,small_max_grounding,large_max_grounding,top1_score,top1_top2_margin
1,1 in 5 million in UK have abnormal PrP positiv...,0.635374,0.451472,0.735898,0.436206,0.019305
6,4-PBA treatment raises endoplasmic reticulum s...,0.654069,0.414013,0.435128,0.567920,0.098642
5,4-PBA treatment decreases endoplasmic reticulu...,0.727522,0.328454,0.437766,0.564231,0.083546
11,61% of colorectal cancer patients are diagnose...,0.760664,0.542656,0.591658,0.661211,0.059509
4,32% of liver transplantation programs required...,0.822267,0.741304,0.681770,0.767422,0.176142
8,5'-nucleotidase metabolizes 6MP.,0.873174,0.512269,0.470042,0.556454,0.147689
0,0-dimensional biomaterials lack inductive prop...,0.878049,0.298222,0.316298,0.345350,0.012000
16,90% of patients with melanoma and an objective...,0.885607,0.652987,0.743028,0.754228,0.162880
2,1-1% of colorectal cancer patients are diagnos...,0.887530,0.668680,0.748001,0.649795,0.046887
10,53% of perinatal mortality is due to low birth...,0.894555,0.511255,0.454046,0.518899,0.010021


In [ ]:
QUALITY_THRESHOLD = 0.85
routing_df["small_model_acceptable"] = (
    routing_df["answer_similarity"] >= QUALITY_THRESHOLD
).astype(int)
print(
    "Quality threshold:",QUALITY_THRESHOLD
)
print(
    "\nAcceptable 20B answers:",routing_df["small_model_acceptable"].sum()
)
print(
    "Queries requiring 120B:",(
        routing_df["small_model_acceptable"] == 0
    ).sum()
)

Quality threshold: 0.85

Acceptable 20B answers: 15
Queries requiring 120B: 5


In [ ]:
router_features = [
    "query_word_count","query_char_count",
    "has_question_mark","top1_score",
    "top2_score","top1_top2_margin",
    "mean_top5_score","std_top5_score"
]
X = routing_df[router_features]
y = routing_df["small_model_acceptable"]
print("Router features:")
for feature in router_features:
    print("-", feature)
print("\nFeature matrix shape:", X.shape)
print("Target shape:", y.shape)

Router features:
- query_word_count
- query_char_count
- has_question_mark
- top1_score
- top2_score
- top1_top2_margin
- mean_top5_score
- std_top5_score

Feature matrix shape: (20, 8)
Target shape: (20,)


In [ ]:
from sklearn.tree import DecisionTreeClassifier
router_model = DecisionTreeClassifier(
    max_depth=3,min_samples_leaf=2,random_state=42
)
router_model.fit(X, y)
print("Router trained successfully.")

Router trained successfully.


In [ ]:
routing_df["router_prediction"] = router_model.predict(X)
routing_df[
    [
        "query","small_model_acceptable","router_prediction"
    ]
].head(10)

,query,small_model_acceptable,router_prediction
0,0-dimensional biomaterials lack inductive prop...,1,1
1,1 in 5 million in UK have abnormal PrP positiv...,0,0
2,1-1% of colorectal cancer patients are diagnos...,1,0
3,10% of sudden infant death syndrome (SIDS) dea...,1,1
4,32% of liver transplantation programs required...,0,0
5,4-PBA treatment decreases endoplasmic reticulu...,0,0
6,4-PBA treatment raises endoplasmic reticulum s...,0,0
7,40mg/day dosage of folic acid and 2mg/day dosa...,1,1
8,5'-nucleotidase metabolizes 6MP.,1,0
9,50% of patients exposed to radiation have acti...,1,1


In [ ]:
routing_counts = routing_df["router_prediction"].value_counts()
print(
    "Queries routed to 20B:",routing_counts.get(1, 0)
)
print(
    "Queries routed to 120B:",routing_counts.get(0, 0)
)

Queries routed to 20B: 13
Queries routed to 120B: 7


In [ ]:
from sklearn.tree import export_text
tree_rules = export_text(
    router_model,feature_names=router_features
)
print(tree_rules)

|--- query_word_count <= 15.00
|   |--- std_top5_score <= 0.03
|   |   |--- class: 1
|   |--- std_top5_score >  0.03
|   |   |--- std_top5_score <= 0.06
|   |   |   |--- class: 0
|   |   |--- std_top5_score >  0.06
|   |   |   |--- class: 0
|--- query_word_count >  15.00
|   |--- class: 1



In [ ]:
from sklearn.metrics import (
    accuracy_score,precision_score,
    recall_score,classification_report
)
y_true = routing_df["small_model_acceptable"]
y_pred = routing_df["router_prediction"]
print(
    "Accuracy:",round(accuracy_score(y_true, y_pred), 3)
)
print(
    "Precision:",round(
        precision_score(
            y_true,y_pred,zero_division=0
        ),3
    )
)
print(
    "Recall:",round(
        recall_score(
            y_true,y_pred,zero_division=0
        ),3
    )
)
print("\nClassification report:\n")
print(
    classification_report(
        y_true,y_pred,zero_division=0
    )
)

Accuracy: 0.9
Precision: 1.0
Recall: 0.867

Classification report:

              precision    recall  f1-score   support

           0       0.71      1.00      0.83         5
           1       1.00      0.87      0.93        15

    accuracy                           0.90        20
   macro avg       0.86      0.93      0.88        20
weighted avg       0.93      0.90      0.90        20



In [ ]:
routing_df["routed_cost"] = np.where(
    routing_df["router_prediction"] == 1,
    routing_df["small_cost"],routing_df["large_cost"]
)
routing_df["baseline_large_cost"] = (
    routing_df["large_cost"]
)
total_baseline_cost = (
    routing_df["baseline_large_cost"].sum()
)
total_routed_cost = (
    routing_df["routed_cost"].sum()
)
cost_saved = (
    total_baseline_cost - total_routed_cost
)
cost_reduction_pct = (
    cost_saved / total_baseline_cost
) * 100
print(
    "Baseline cost using only 120B: $",round(total_baseline_cost, 6)
)
print(
    "Router cost: $",round(total_routed_cost, 6)
)
print(
    "Estimated cost saved: $",round(cost_saved, 6)
)
print(
    "Estimated cost reduction:",round(cost_reduction_pct, 2),"%"
)

Baseline cost using only 120B: $ 0.008544
Router cost: $ 0.005813
Estimated cost saved: $ 0.002731
Estimated cost reduction: 31.97 %


In [ ]:
small_only_cost = routing_df["small_cost"].sum()
large_only_cost = routing_df["large_cost"].sum()
router_cost = routing_df["routed_cost"].sum()
strategy_comparison = pd.DataFrame({
    "Strategy": [
        "Always 20B","Always 120B","Intelligent Router"
    ],
    "Total Cost": [
        small_only_cost,large_only_cost,router_cost
    ]
})
strategy_comparison["Cost Reduction vs 120B (%)"] = (
    1 - strategy_comparison["Total Cost"] / large_only_cost) * 100
strategy_comparison

,Strategy,Total Cost,Cost Reduction vs 120B (%)
0,Always 20B,0.004265,50.080757
1,Always 120B,0.008544,0.000000
2,Intelligent Router,0.005813,31.969242


In [ ]:
routed_to_small = (routing_df["router_prediction"] == 1)
small_was_acceptable = (routing_df["small_model_acceptable"] == 1)
correct_small_routes = (
    routed_to_small & small_was_acceptable
).sum()
incorrect_small_routes = (
    routed_to_small & ~small_was_acceptable
).sum()
total_small_routes = routed_to_small.sum()
print(
    "Total routed to 20B:",total_small_routes
)
print(
    "Correct 20B routes:",correct_small_routes
)
print(
    "Potentially unsafe 20B routes:", incorrect_small_routes
)

Total routed to 20B: 13
Correct 20B routes: 13
Potentially unsafe 20B routes: 0


In [ ]:
semantic_cache = []
CACHE_THRESHOLD = 0.90
print("Semantic cache initialized.")
print("Cache similarity threshold:", CACHE_THRESHOLD)

Semantic cache initialized.
Cache similarity threshold: 0.9


In [ ]:
def find_cache_match(query, threshold=CACHE_THRESHOLD):
    if len(semantic_cache) == 0:
        return None
    query_embedding = embedding_model.encode(
        [query],normalize_embeddings=True
    ).astype("float32")
    best_match = None
    best_similarity = -1
    for entry in semantic_cache:
        similarity = float(
            np.dot(
                query_embedding[0],entry["embedding"]
            )
        )
        if similarity > best_similarity:
            best_similarity = similarity
            best_match = entry
    if best_similarity >= threshold:
        return {
            "answer": best_match["answer"],
            "cached_query": best_match["query"],
            "similarity": best_similarity
        }
    return None

In [ ]:
def add_to_cache(query, answer):
    query_embedding = embedding_model.encode(
        [query],normalize_embeddings=True
    ).astype("float32")[0]
    semantic_cache.append({
        "query": query,"answer": answer,"embedding": query_embedding
    })

In [ ]:
def route_query(query):
    features = extract_query_features(
        query,top_k=5
    )
    feature_vector = pd.DataFrame(
        [features]
    )[router_features]
    prediction = router_model.predict(
        feature_vector
    )[0]
    if prediction == 1:
        selected_model = SMALL_MODEL
    else:
        selected_model = LARGE_MODEL
    return {
        "selected_model": selected_model,
        "prediction": int(prediction),
        "features": features
    }

In [ ]:
def cost_aware_rag(
    query,top_k=5,cache_threshold=CACHE_THRESHOLD
):
    start_time = time.perf_counter()
    cache_match = find_cache_match(
        query,threshold=cache_threshold
    )
    if cache_match is not None:
        end_time = time.perf_counter()
        return {
            "query": query,"cache_hit": True,
            "cache_similarity": cache_match["similarity"],
            "cached_query": cache_match["cached_query"],
            "selected_model": "CACHE",
            "answer": cache_match["answer"],
            "input_tokens": 0,
            "output_tokens": 0,
            "total_tokens": 0,
            "estimated_cost": 0.0,
            "latency_sec": end_time - start_time
        }
    routing_result = route_query(query)
    selected_model = routing_result["selected_model"]
    result = generate_rag_answer(
        query=query,model=selected_model,top_k=top_k
    )
    estimated_cost = calculate_model_cost(
        selected_model,result["input_tokens"],result["output_tokens"]
    )
    add_to_cache(
        query,result["answer"]
    )
    end_time = time.perf_counter()
    return {
        "query": query,"cache_hit": False,
        "cache_similarity": None,"cached_query": None,
        "selected_model": selected_model,
        "answer": result["answer"],
        "input_tokens": result["input_tokens"],
        "output_tokens": result["output_tokens"],
        "total_tokens": result["total_tokens"],
        "estimated_cost": estimated_cost,
        "latency_sec": end_time - start_time
    }

In [ ]:
result_cached = cost_aware_rag(
    test_query,
    top_k=5
)
print("Query:", result_cached["query"])
print("Cache hit:", result_cached["cache_hit"])
print("Selected model:", result_cached["selected_model"])
print("Cache similarity:", result_cached["cache_similarity"])
print("Estimated cost:", result_cached["estimated_cost"])
print("Latency:", round(result_cached["latency_sec"], 3), "seconds")

Query: 0-dimensional biomaterials lack inductive properties.
Cache hit: False
Selected model: openai/gpt-oss-20b
Cache similarity: None
Estimated cost: 0.00011812500000000001
Latency: 0.973 seconds


In [ ]:
workload_queries = benchmark_queries[:10] * 2
print("Total requests:", len(workload_queries))
print("Unique queries:", len(set(workload_queries)))

Total requests: 20
Unique queries: 10


In [ ]:
semantic_cache = []
print("Cache reset.")

Cache reset.


In [ ]:
end_to_end_results = []
for i, query in enumerate(
    workload_queries,start=1
):
    print(f"Processing {i}/{len(workload_queries)}")
    result = cost_aware_rag(query=query,top_k=5)
    end_to_end_results.append(result)

Processing 1/20
Processing 2/20
Processing 3/20
Processing 4/20
Processing 5/20
Processing 6/20
Processing 7/20
Processing 8/20
Processing 9/20
Processing 10/20
Processing 11/20
Processing 12/20
Processing 13/20
Processing 14/20
Processing 15/20
Processing 16/20
Processing 17/20
Processing 18/20
Processing 19/20
Processing 20/20


In [ ]:
end_to_end_df = pd.DataFrame(end_to_end_results)
end_to_end_df[
    [
        "query","cache_hit","selected_model","estimated_cost","latency_sec"
    ]
]

,query,cache_hit,selected_model,estimated_cost,latency_sec
0,0-dimensional biomaterials lack inductive prop...,False,openai/gpt-oss-20b,0.000118,0.747941
1,1 in 5 million in UK have abnormal PrP positiv...,False,openai/gpt-oss-120b,0.000526,1.385368
2,1-1% of colorectal cancer patients are diagnos...,False,openai/gpt-oss-120b,0.000451,1.283849
3,10% of sudden infant death syndrome (SIDS) dea...,False,openai/gpt-oss-20b,0.000206,0.937301
4,32% of liver transplantation programs required...,False,openai/gpt-oss-120b,0.000465,1.426188
5,4-PBA treatment decreases endoplasmic reticulu...,False,openai/gpt-oss-120b,0.000402,4.060843
6,4-PBA treatment raises endoplasmic reticulum s...,True,CACHE,0.000000,0.027338
7,40mg/day dosage of folic acid and 2mg/day dosa...,False,openai/gpt-oss-20b,0.000274,1.138927
8,5'-nucleotidase metabolizes 6MP.,False,openai/gpt-oss-120b,0.000411,15.318937
9,50% of patients exposed to radiation have acti...,False,openai/gpt-oss-20b,0.000171,1.327063


In [ ]:
total_requests = len(end_to_end_df)
cache_hits = (end_to_end_df["cache_hit"].sum())
cache_hit_rate = (
    cache_hits / total_requests
) * 100
llm_calls = (~end_to_end_df["cache_hit"]).sum()
small_model_calls = (
    (
        end_to_end_df["selected_model"] == SMALL_MODEL
    ).sum()
)
large_model_calls = (
    (
        end_to_end_df["selected_model"] == LARGE_MODEL
    ).sum()
)
total_cost = (
    end_to_end_df["estimated_cost"].sum()
)
average_latency = (end_to_end_df["latency_sec"].mean())
print("Total requests:", total_requests)
print("Cache hits:", cache_hits)
print("Cache hit rate:", round(cache_hit_rate, 2), "%")
print("LLM calls:", llm_calls)
print("20B calls:", small_model_calls)
print("120B calls:", large_model_calls)
print("Total estimated cost: $", round(total_cost, 6))
print("Average latency:",round(average_latency, 3),"seconds")

Total requests: 20
Cache hits: 11
Cache hit rate: 55.0 %
LLM calls: 9
20B calls: 4
120B calls: 5
Total estimated cost: $ 0.003025
Average latency: 1.405 seconds


In [ ]:
baseline_large_cost = (
    benchmark_df[benchmark_df["query"].isin(workload_queries)]["large_cost"].sum()
)
baseline_large_cost *= 2
cost_saved = (
    baseline_large_cost - total_cost
)
cost_reduction = (
    cost_saved / baseline_large_cost
) * 100
print(
    "120B-only baseline cost: $",round(baseline_large_cost, 6)
)
print(
    "Cache + Router cost: $", round(total_cost, 6)
)
print(
    "Estimated cost saved: $",round(cost_saved, 6)
)
print(
    "Estimated cost reduction:",round(cost_reduction, 2),"%"
)

120B-only baseline cost: $ 0.008408
Cache + Router cost: $ 0.003025
Estimated cost saved: $ 0.005383
Estimated cost reduction: 64.03 %


In [ ]:
end_to_end_df.to_csv(
    "cost_aware_rag_end_to_end.csv",index=False
)
strategy_comparison.to_csv(
    "routing_strategy_comparison.csv",index=False
)
routing_df.to_csv(
    "routing_evaluation_dataset.csv",index=False
)
print("End-to-end experiment files saved.")

End-to-end experiment files saved.
